# Prétraitement et encodage des données


## Introduction





Le prétraitement des données constitue une étape essentielle du processus de Data Mining. Il consiste à transformer les données nettoyées afin de les rendre exploitables par les algorithmes d'apprentissage automatique.

Dans ce notebook, nous allons réaliser les opérations suivantes :

- Le traitement des valeurs manquantes par des méthodes d'imputation adaptées.
- L'encodage des variables catégorielles.
- La normalisation ou la standardisation des variables numériques, selon les besoins.
- La préparation du jeu de données pour la phase de modélisation.

Afin d'éviter toute fuite de données (*Data Leakage*), les transformations seront ajustées uniquement sur les données d'entraînement, puis appliquées aux données de validation et de test.

La variable cible de notre étude est `Credit_Score`, qui comporte trois classes : `Poor`, `Standard` et `Good`.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement du dataset nettoyé
train_clean = pd.read_csv("train_clean.csv")

# Vérification des dimensions
print("Dimensions du dataset :", train_clean.shape)

# Affichage des premières observations
display(train_clean.head())

# Vérification des types de données
print("\nTypes des variables :")
print(train_clean.dtypes)

Dimensions du dataset : (100000, 28)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23.0,821-00-0265,Scientist,19114.12,1824.843333,3.0,...,NaN,809.98,26.822620,265.0,No,49.574949,80.415295,High_spent_Small_value_payments,312.494089,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23.0,821-00-0265,Scientist,19114.12,NaN,3.0,...,Good,809.98,31.944960,NaN,No,49.574949,118.280222,Low_spent_Large_value_payments,284.629162,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,NaN,821-00-0265,Scientist,19114.12,NaN,3.0,...,Good,809.98,28.609352,267.0,No,49.574949,81.699521,Low_spent_Medium_value_payments,331.209863,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23.0,821-00-0265,Scientist,19114.12,NaN,3.0,...,Good,809.98,31.377862,268.0,No,49.574949,199.458074,Low_spent_Small_value_payments,223.451310,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23.0,821-00-0265,Scientist,19114.12,1824.843333,3.0,...,Good,809.98,24.797347,269.0,No,49.574949,41.420153,High_spent_Medium_value_payments,341.489231,Good



Types des variables :
ID                              str
Customer_ID                     str
Month                           str
Name                            str
Age                         float64
SSN                             str
Occupation                      str
Annual_Income               float64
Monthly_Inhand_Salary       float64
Num_Bank_Accounts           float64
Num_Credit_Card             float64
Interest_Rate               float64
Num_of_Loan                 float64
Type_of_Loan                    str
Delay_from_due_date           int64
Num_of_Delayed_Payment      float64
Changed_Credit_Limit        float64
Num_Credit_Inquiries        float64
Credit_Mix                      str
Outstanding_Debt            float64
Credit_Utilization_Ratio    float64
Credit_History_Age          float64
Payment_of_Min_Amount           str
Total_EMI_per_month         float64
Amount_invested_monthly     float64
Payment_Behaviour               str
Monthly_Balance             float64
Credi


### 1.1. Identification des valeurs manquantes

Après la phase de nettoyage, certaines valeurs manquantes subsistent dans le jeu de données.

Ces valeurs peuvent provenir de données initialement absentes ou de valeurs invalides qui ont été remplacées par `NaN` durant le nettoyage.

Avant de procéder à leur imputation, nous allons identifier les variables concernées, calculer le nombre de valeurs manquantes ainsi que leur proportion par rapport au nombre total d'observations.

Cette analyse permettra de mieux comprendre la répartition des valeurs manquantes et de choisir les méthodes d'imputation appropriées.


In [2]:
# Calcul du nombre de valeurs manquantes par variable
missing_count = train_clean.isnull().sum()

# Calcul du pourcentage de valeurs manquantes
missing_percentage = (missing_count / len(train_clean)) * 100

# Création d'un tableau récapitulatif
missing_summary = pd.DataFrame({
    "Nombre de valeurs manquantes": missing_count,
    "Pourcentage (%)": missing_percentage
})

# Conservation uniquement des variables contenant des NaN
missing_summary = missing_summary[missing_summary["Nombre de valeurs manquantes"] > 0]

# Tri par ordre décroissant
missing_summary = missing_summary.sort_values(
    by="Pourcentage (%)",
    ascending=False
)

display(missing_summary.round(2))

,Nombre de valeurs manquantes,Pourcentage (%)
Credit_Mix,20195,20.20
Monthly_Inhand_Salary,15002,15.00
Type_of_Loan,11408,11.41
Name,9985,9.98
Credit_History_Age,9030,9.03
Amount_invested_monthly,8784,8.78
Num_of_Delayed_Payment,8382,8.38
Payment_Behaviour,7600,7.60
Occupation,7062,7.06
Num_of_Loan,4348,4.35



## 2. Sélection des variables explicatives et de la variable cible

Le jeu de données contient des variables descriptives, des identifiants et une variable cible.

Pour préparer les données à la modélisation, nous allons :

- Exclure les identifiants (`ID`, `Customer_ID`, `Name` et `SSN`), car ils ne constituent pas des caractéristiques pertinentes pour la prédiction.
- Séparer `Credit_Score`, qui représente la variable cible à prédire.
- Conserver les autres variables comme variables explicatives.

Cette séparation permettra de préparer indépendamment les caractéristiques d'entrée et la variable à prédire.


In [3]:
# Définition des variables à exclure
columns_to_drop = ["ID", "Customer_ID", "Name", "SSN"]

# Séparation des variables explicatives et de la variable cible
X = train_clean.drop(columns=columns_to_drop + ["Credit_Score"])
y = train_clean["Credit_Score"]

# Vérification
print("Dimensions des variables explicatives :", X.shape)
print("Dimensions de la variable cible :", y.shape)

print("\nRépartition des classes :")
display(y.value_counts())

Dimensions des variables explicatives : (100000, 23)
Dimensions de la variable cible : (100000,)

Répartition des classes :


Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64


### 2.1. Division du jeu de données

Afin d'évaluer correctement les performances des modèles, nous divisons le jeu de données en deux sous-ensembles :

- Un ensemble d'entraînement représentant 80 % des observations.
- Un ensemble de validation représentant 20 % des observations.

Nous utilisons une séparation stratifiée (`stratify`) pour préserver la distribution des classes de la variable cible dans les deux ensembles.

La graine aléatoire (`random_state=42`) garantit la reproductibilité de cette séparation.


In [5]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 996.7 kB/s eta 0:00:08
   --- ------------------------------------ 0.8/8.4 MB 1.1 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.4 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.4 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.4 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.4 MB 1.1 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.4 MB 675.5 kB/s eta 0:00:11
   ------ --------------------------------- 1.3/8.4 MB 675.5 kB/s eta 0:00:11
   ------- ---------------------

In [6]:
from sklearn.model_selection import train_test_split

# Division des données
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Vérification des dimensions
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_val   :", y_val.shape)

# Vérification de la répartition des classes
print("\nRépartition des classes dans l'ensemble d'entraînement :")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nRépartition des classes dans l'ensemble de validation :")
display((y_val.value_counts(normalize=True) * 100).round(2))

Dimensions de X_train : (80000, 23)
Dimensions de X_val   : (20000, 23)
Dimensions de y_train : (80000,)
Dimensions de y_val   : (20000,)

Répartition des classes dans l'ensemble d'entraînement :


Credit_Score
Standard    53.17
Poor        29.00
Good        17.83
Name: proportion, dtype: float64


Répartition des classes dans l'ensemble de validation :


Credit_Score
Standard    53.18
Poor        28.99
Good        17.83
Name: proportion, dtype: float64


### 2.2. Identification des variables numériques et catégorielles

Les variables explicatives ne sont pas toutes de même nature.

Nous distinguons deux catégories :

- Les variables numériques, qui représentent des mesures ou des quantités.
- Les variables catégorielles, qui représentent des catégories ou des modalités.

Cette distinction est nécessaire, car les méthodes d'imputation seront adaptées au type de chaque variable.

L'identification est effectuée sur l'ensemble d'entraînement afin de préparer les transformations sans utiliser les données de validation.


In [7]:
# Identification des variables numériques
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identification des variables catégorielles
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Nombre de variables numériques :", len(numerical_features))
print("Variables numériques :")
print(numerical_features)

print("\nNombre de variables catégorielles :", len(categorical_features))
print("Variables catégorielles :")
print(categorical_features)

Nombre de variables numériques : 17
Variables numériques :
['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

Nombre de variables catégorielles : 6
Variables catégorielles :
['Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']


C:\Users\user\AppData\Local\Temp\ipykernel_3668\2264871313.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(



### 2.3. Choix des stratégies d'imputation

Les valeurs manquantes seront traitées selon la nature des variables :

- **Variables numériques :** nous utiliserons la médiane. Cette statistique est moins influencée par les valeurs extrêmes que la moyenne.
- **Variables catégorielles :** nous remplacerons les valeurs manquantes par une modalité spécifique, « Inconnu », afin de conserver l'information liée à leur absence.

Les paramètres d'imputation seront calculés exclusivement à partir de l'ensemble d'entraînement, puis appliqués à l'ensemble de validation.

Cette démarche permet d'éviter toute fuite de données pendant le prétraitement.


In [8]:
# Valeurs manquantes dans les variables numériques
print("Valeurs manquantes dans les variables numériques :")
display(X_train[numerical_features].isnull().sum().sort_values(ascending=False))

# Valeurs manquantes dans les variables catégorielles
print("\nValeurs manquantes dans les variables catégorielles :")
display(X_train[categorical_features].isnull().sum().sort_values(ascending=False))

Valeurs manquantes dans les variables numériques :


Monthly_Inhand_Salary       12032
Credit_History_Age           7240
Amount_invested_monthly      7084
Num_of_Delayed_Payment       6700
Num_of_Loan                  3517
Num_Credit_Inquiries         2891
Total_EMI_per_month          2270
Age                          2206
Num_Credit_Card              1776
Changed_Credit_Limit         1691
Interest_Rate                1613
Num_Bank_Accounts            1073
Monthly_Balance               957
Annual_Income                 717
Delay_from_due_date             0
Credit_Utilization_Ratio        0
Outstanding_Debt                0
dtype: int64


Valeurs manquantes dans les variables catégorielles :


Credit_Mix               16108
Type_of_Loan              9157
Payment_Behaviour         6098
Occupation                5691
Month                        0
Payment_of_Min_Amount        0
dtype: int64


### 2.4. Imputation des valeurs manquantes

L'imputation consiste à remplacer les valeurs manquantes par des valeurs estimées afin de permettre l'exploitation des données par les algorithmes d'apprentissage automatique.

Nous utilisons :

- La médiane pour les variables numériques, afin de limiter l'influence des valeurs extrêmes.
- La modalité « Inconnu » pour les variables catégorielles, afin de conserver l'information relative aux données absentes.

Les paramètres d'imputation sont appris sur l'ensemble d'entraînement, puis appliqués à l'ensemble de validation.


In [9]:
from sklearn.impute import SimpleImputer

# Imputeur pour les variables numériques
numerical_imputer = SimpleImputer(strategy="median")

# Imputeur pour les variables catégorielles
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Inconnu"
)

In [10]:
# Imputation des variables numériques
X_train[numerical_features] = numerical_imputer.fit_transform(
    X_train[numerical_features]
)

X_val[numerical_features] = numerical_imputer.transform(
    X_val[numerical_features]
)

# Imputation des variables catégorielles
X_train[categorical_features] = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

X_val[categorical_features] = categorical_imputer.transform(
    X_val[categorical_features]
)


Après l'application des stratégies d'imputation, nous vérifions que les ensembles d'entraînement et de validation ne contiennent plus de valeurs manquantes.

Cette vérification permet de confirmer que le traitement a été correctement effectué.


In [11]:
print("Valeurs manquantes dans X_train :", X_train.isnull().sum().sum())
print("Valeurs manquantes dans X_val   :", X_val.isnull().sum().sum())

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_val   : 0



### 2.5. Analyse des variables catégorielles avant encodage

Les algorithmes d'apprentissage automatique ne peuvent généralement pas exploiter directement les valeurs textuelles.

Avant de procéder à l'encodage, nous étudions les modalités distinctes de chaque variable catégorielle.

Cette analyse permettra de choisir une méthode d'encodage adaptée à la nature et au nombre de modalités de chaque variable.


In [12]:
for col in categorical_features:
    print(f"\nVariable : {col}")
    print("Nombre de modalités :", X_train[col].nunique())
    print("Modalités :", X_train[col].unique())


Variable : Month
Nombre de modalités : 8
Modalités : <StringArray>
['May', 'August', 'February', 'March', 'June', 'July', 'April', 'January']
Length: 8, dtype: str

Variable : Occupation
Nombre de modalités : 16
Modalités : <StringArray>
[     'Musician',        'Writer',        'Doctor',    'Accountant',
       'Inconnu',      'Engineer',       'Manager', 'Media_Manager',
       'Teacher',     'Developer',     'Architect',      'Mechanic',
        'Lawyer',  'Entrepreneur',     'Scientist',    'Journalist']
Length: 16, dtype: str

Variable : Type_of_Loan
Nombre de modalités : 4310
Modalités : <StringArray>
[                                                   'Payday Loan, Student Loan, Debt Consolidation Loan',
                                                                                         'Mortgage Loan',
                                                                                      'Home Equity Loan',
                                                     'Personal Loa


#### 2.5.1. Encodage cyclique de la variable Month

La variable `Month` représente le mois auquel correspond chaque observation. Les mois possèdent un ordre chronologique cyclique : le mois de décembre est suivi du mois de janvier.

Un encodage numérique classique ne permettrait pas de représenter correctement cette relation.

Nous utilisons donc un encodage cyclique basé sur les fonctions sinus et cosinus. Chaque mois sera représenté par deux nouvelles variables :

- `Month_sin`
- `Month_cos`

Cette représentation permet de préserver la proximité entre les mois situés de part et d'autre de la fin de l'année.


In [13]:
# Association de chaque mois à son numéro
month_mapping = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

# Conversion des mois en valeurs numériques
month_train = X_train["Month"].map(month_mapping)
month_val = X_val["Month"].map(month_mapping)

# Vérification des correspondances
assert month_train.notna().all()
assert month_val.notna().all()

# Encodage cyclique
X_train["Month_sin"] = np.sin(2 * np.pi * (month_train - 1) / 12)
X_train["Month_cos"] = np.cos(2 * np.pi * (month_train - 1) / 12)

X_val["Month_sin"] = np.sin(2 * np.pi * (month_val - 1) / 12)
X_val["Month_cos"] = np.cos(2 * np.pi * (month_val - 1) / 12)

# Suppression de la variable originale
X_train.drop(columns=["Month"], inplace=True)
X_val.drop(columns=["Month"], inplace=True)

# Mise à jour de la liste des variables catégorielles
categorical_features.remove("Month")

# Vérification
display(X_train[["Month_sin", "Month_cos"]].head())

print("Nombre de variables catégorielles restantes :",
      len(categorical_features))

,Month_sin,Month_cos
10268,0.866025,-0.500000
12727,-0.500000,-0.866025
30953,0.500000,0.866025
61394,0.866025,0.500000
90061,0.500000,-0.866025


Nombre de variables catégorielles restantes : 5


In [14]:
display(
    X_train[["Month_sin", "Month_cos"]]
    .head()
    .reset_index(drop=True)
)

,Month_sin,Month_cos
0,0.866025,-0.500000
1,-0.500000,-0.866025
2,0.500000,0.866025
3,0.866025,0.500000
4,0.500000,-0.866025



#### 2.5.2. Encodage One-Hot de la variable Occupation

La variable `Occupation` représente la profession du client. Elle comporte plusieurs modalités catégorielles sans ordre hiérarchique.

Nous utilisons le One-Hot Encoding pour transformer chaque profession en une variable binaire.

L'encodeur est ajusté sur l'ensemble d'entraînement, puis appliqué à l'ensemble de validation. Le paramètre `handle_unknown="ignore"` permet de gérer les catégories inconnues rencontrées lors de la validation.


In [15]:
from sklearn.preprocessing import OneHotEncoder

# Création de l'encodeur
occupation_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apprentissage et transformation sur l'entraînement
occupation_train_encoded = occupation_encoder.fit_transform(
    X_train[["Occupation"]]
)

# Transformation de la validation
occupation_val_encoded = occupation_encoder.transform(
    X_val[["Occupation"]]
)

# Récupération des noms des nouvelles colonnes
occupation_columns = occupation_encoder.get_feature_names_out(
    ["Occupation"]
)

print("Nombre de nouvelles colonnes :", len(occupation_columns))
print("Noms des colonnes :")
print(occupation_columns)

Nombre de nouvelles colonnes : 16
Noms des colonnes :
['Occupation_Accountant' 'Occupation_Architect' 'Occupation_Developer'
 'Occupation_Doctor' 'Occupation_Engineer' 'Occupation_Entrepreneur'
 'Occupation_Inconnu' 'Occupation_Journalist' 'Occupation_Lawyer'
 'Occupation_Manager' 'Occupation_Mechanic' 'Occupation_Media_Manager'
 'Occupation_Musician' 'Occupation_Scientist' 'Occupation_Teacher'
 'Occupation_Writer']



##### Intégration des variables encodées

Après avoir ajusté l'encodeur sur l'ensemble d'entraînement, nous intégrons les nouvelles variables binaires dans les deux ensembles de données.

La variable catégorielle originale `Occupation` est ensuite supprimée, puisqu'elle est désormais représentée par les colonnes numériques générées par le One-Hot Encoding.


In [16]:
# Création des DataFrames encodés
occupation_train_df = pd.DataFrame(
    occupation_train_encoded,
    columns=occupation_columns,
    index=X_train.index
)

occupation_val_df = pd.DataFrame(
    occupation_val_encoded,
    columns=occupation_columns,
    index=X_val.index
)

# Intégration des nouvelles colonnes
X_train = pd.concat(
    [X_train.drop(columns=["Occupation"]), occupation_train_df],
    axis=1
)

X_val = pd.concat(
    [X_val.drop(columns=["Occupation"]), occupation_val_df],
    axis=1
)

# Mise à jour de la liste des variables catégorielles
categorical_features.remove("Occupation")

# Vérification
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)
print("Nombre de variables catégorielles restantes :",
      len(categorical_features))

display(X_train[occupation_columns].head())

Dimensions de X_train : (80000, 39)
Dimensions de X_val   : (20000, 39)
Nombre de variables catégorielles restantes : 4


,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Inconnu,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer
10268,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
12727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
30953,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
61394,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90061,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# Vérifier que chaque observation possède exactement une profession
occupation_sum = X_train[occupation_columns].sum(axis=1)

print("Nombre de lignes avec une seule profession :",
      (occupation_sum == 1).sum())

print("Nombre de lignes avec zéro ou plusieurs professions :",
      (occupation_sum != 1).sum())

Nombre de lignes avec une seule profession : 80000
Nombre de lignes avec zéro ou plusieurs professions : 0



#### 2.5.3. Analyse de la variable Type_of_Loan

La variable `Type_of_Loan` est une variable catégorielle multivaluée. Une même observation peut contenir plusieurs types de prêts séparés par des virgules.

Un encodage One-Hot classique n'est pas adapté, car il traiterait chaque combinaison de prêts comme une catégorie différente.

Nous allons utiliser un encodage Multi-Hot afin de représenter chaque type de prêt par une variable binaire indépendante.


In [18]:
# Examiner quelques combinaisons de prêts
display(X_train["Type_of_Loan"].value_counts().head(15))

# Examiner des exemples de valeurs
display(X_train["Type_of_Loan"].head(10))

Type_of_Loan
Inconnu                                 9157
Not Specified                           1232
Student Loan                            1207
Credit-Builder Loan                     1197
Personal Loan                           1189
Debt Consolidation Loan                 1167
Payday Loan                             1129
Mortgage Loan                           1126
Auto Loan                               1084
Home Equity Loan                        1022
Personal Loan, Student Loan              327
Not Specified, Payday Loan               307
Credit-Builder Loan, Not Specified       290
Payday Loan, Debt Consolidation Loan     287
Credit-Builder Loan, Mortgage Loan       284
Name: count, dtype: int64

10268    Payday Loan, Student Loan, Debt Consolidation ...
12727                                        Mortgage Loan
30953                                     Home Equity Loan
61394    Personal Loan, Mortgage Loan, Credit-Builder Loan
90061    Student Loan, Credit-Builder Loan, Debt Consol...
25858        Student Loan, Personal Loan, Home Equity Loan
81010    Payday Loan, Debt Consolidation Loan, Not Spec...
16546    Credit-Builder Loan, Home Equity Loan, Not Spe...
2855                                           Payday Loan
38341    Credit-Builder Loan, Home Equity Loan, Not Spe...
Name: Type_of_Loan, dtype: str


##### Identification des types de prêts

Avant l'encodage, nous identifions les différents types de prêts présents dans les observations.

Les combinaisons sont séparées à partir des virgules, puis chaque élément est nettoyé des espaces superflus.

La modalité `Inconnu` est exclue de cette extraction, car elle représente une information manquante et non un type de prêt.


In [19]:
# Extraire les types de prêts individuels
loan_types = set()

for value in X_train["Type_of_Loan"]:
    if value != "Inconnu":
        loans = value.split(",")
        for loan in loans:
            loan_types.add(loan.strip())

# Afficher les types identifiés
loan_types = sorted(loan_types)

print("Nombre de types de prêts individuels :", len(loan_types))
print("\nTypes de prêts identifiés :")
print(loan_types)

Nombre de types de prêts individuels : 9

Types de prêts identifiés :
['Auto Loan', 'Credit-Builder Loan', 'Debt Consolidation Loan', 'Home Equity Loan', 'Mortgage Loan', 'Not Specified', 'Payday Loan', 'Personal Loan', 'Student Loan']



##### Application du Multi-Hot Encoding

Chaque type de prêt est représenté par une colonne binaire :

- `1` : le client possède ce type de prêt.
- `0` : le client ne possède pas ce type de prêt.

Une colonne supplémentaire `Type_of_Loan_Inconnu` indique l'absence d'information sur les prêts du client.

L'encodage est construit à partir des catégories de l'ensemble d'entraînement, puis appliqué à l'ensemble de validation avec les mêmes colonnes.


In [20]:
# Fonction de transformation Multi-Hot
def encode_loans(data, loan_types):
    encoded = pd.DataFrame(
        0,
        index=data.index,
        columns=loan_types + ["Type_of_Loan_Inconnu"]
    )

    for index, value in data.items():
        if value == "Inconnu":
            encoded.loc[index, "Type_of_Loan_Inconnu"] = 1
        else:
            loans = [loan.strip() for loan in value.split(",")]

            for loan in loans:
                if loan in loan_types:
                    encoded.loc[index, loan] = 1

    return encoded


# Encodage de l'entraînement et de la validation
loan_train_encoded = encode_loans(X_train["Type_of_Loan"], loan_types)
loan_val_encoded = encode_loans(X_val["Type_of_Loan"], loan_types)

# Vérification des dimensions et des colonnes
print("Dimensions entraînement :", loan_train_encoded.shape)
print("Dimensions validation   :", loan_val_encoded.shape)

display(loan_train_encoded.head())

Dimensions entraînement : (80000, 10)
Dimensions validation   : (20000, 10)


,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,Not Specified,Payday Loan,Personal Loan,Student Loan,Type_of_Loan_Inconnu
10268,0,0,1,0,0,0,1,0,1,0
12727,0,0,0,0,1,0,0,0,0,0
30953,0,0,0,1,0,0,0,0,0,0
61394,0,1,0,0,1,0,0,1,0,0
90061,0,1,1,0,0,0,0,1,1,0


In [21]:
# Nombre de types de prêts attribués à chaque client
loan_counts = loan_train_encoded[loan_types].sum(axis=1)

# Vérifications
print("Clients sans type de prêt identifié :",
      (loan_train_encoded.sum(axis=1) == 0).sum())

print("Clients avec plusieurs types de prêts :",
      (loan_counts > 1).sum())

print("Clients avec une information inconnue :",
      loan_train_encoded["Type_of_Loan_Inconnu"].sum())

print("Clients Inconnu avec un autre prêt activé :",
      (
          (loan_train_encoded["Type_of_Loan_Inconnu"] == 1)
          & (loan_counts > 0)
      ).sum())

Clients sans type de prêt identifié : 0
Clients avec plusieurs types de prêts : 60490
Clients avec une information inconnue : 9157
Clients Inconnu avec un autre prêt activé : 0



##### Intégration des variables Multi-Hot

Les 10 variables binaires créées sont intégrées aux ensembles d'entraînement et de validation.

La colonne originale `Type_of_Loan` est supprimée, car son information est désormais représentée par les nouvelles variables binaires.


In [22]:
# Intégration des variables encodées
X_train = pd.concat(
    [X_train.drop(columns=["Type_of_Loan"]), loan_train_encoded],
    axis=1
)

X_val = pd.concat(
    [X_val.drop(columns=["Type_of_Loan"]), loan_val_encoded],
    axis=1
)

# Mise à jour des variables catégorielles
categorical_features.remove("Type_of_Loan")

# Vérification
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)
print("Variables catégorielles restantes :", categorical_features)

Dimensions de X_train : (80000, 48)
Dimensions de X_val   : (20000, 48)
Variables catégorielles restantes : ['Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']



#### 2.5.4. Encodage ordinal de Credit_Mix

La variable `Credit_Mix` possède trois modalités ordonnées : `Bad`, `Standard` et `Good`.

Nous utilisons un encodage ordinal pour préserver cet ordre. La modalité `Inconnu` reçoit une valeur distincte afin de signaler l'absence d'information.


In [23]:
print("Modalités de Credit_Mix dans l'entraînement :")
print(X_train["Credit_Mix"].value_counts())

print("\nModalités de Credit_Mix dans la validation :")
print(X_val["Credit_Mix"].value_counts())

Modalités de Credit_Mix dans l'entraînement :
Credit_Mix
Standard    29214
Good        19455
Inconnu     16108
Bad         15223
Name: count, dtype: int64

Modalités de Credit_Mix dans la validation :
Credit_Mix
Standard    7265
Good        4882
Inconnu     4087
Bad         3766
Name: count, dtype: int64



##### Transformation des modalités de Credit_Mix

Les modalités de `Credit_Mix` sont remplacées par des valeurs numériques selon le mapping défini précédemment.

Le même mapping est appliqué aux ensembles d'entraînement et de validation afin de garantir une représentation cohérente.


In [25]:
# Définition du mapping ordinal
credit_mix_mapping = {
    "Inconnu": -1,
    "Bad": 0,
    "Standard": 1,
    "Good": 2
}

# Application du mapping
X_train["Credit_Mix"] = X_train["Credit_Mix"].map(credit_mix_mapping)
X_val["Credit_Mix"] = X_val["Credit_Mix"].map(credit_mix_mapping)

# Vérification
print("Modalités encodées dans l'entraînement :")
print(X_train["Credit_Mix"].value_counts().sort_index())

print("\nModalités encodées dans la validation :")
print(X_val["Credit_Mix"].value_counts().sort_index())

Modalités encodées dans l'entraînement :
Credit_Mix
-1    16108
 0    15223
 1    29214
 2    19455
Name: count, dtype: int64

Modalités encodées dans la validation :
Credit_Mix
-1    4087
 0    3766
 1    7265
 2    4882
Name: count, dtype: int64



#### 2.5.5. Encodage One-Hot de Payment_of_Min_Amount

La variable `Payment_of_Min_Amount` comporte trois modalités : `Yes`, `No` et `NM`.

Nous utilisons le One-Hot Encoding pour représenter chaque modalité par une variable binaire indépendante, sans supposer de relation d'ordre entre elles.


In [26]:
print("Modalités dans l'entraînement :")
print(X_train["Payment_of_Min_Amount"].value_counts())

print("\nModalités dans la validation :")
print(X_val["Payment_of_Min_Amount"].value_counts())

Modalités dans l'entraînement :
Payment_of_Min_Amount
Yes    41857
No     28509
NM      9634
Name: count, dtype: int64

Modalités dans la validation :
Payment_of_Min_Amount
Yes    10469
No      7158
NM      2373
Name: count, dtype: int64



##### Transformation de Payment_of_Min_Amount

Nous utilisons `OneHotEncoder` pour créer une variable binaire pour chaque modalité.

L'encodeur est ajusté sur l'ensemble d'entraînement, puis utilisé pour transformer l'ensemble de validation afin de garantir la cohérence des colonnes.


In [27]:
# Création de l'encodeur
payment_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apprentissage et transformation
payment_train_encoded = payment_encoder.fit_transform(
    X_train[["Payment_of_Min_Amount"]]
)

# Transformation de la validation
payment_val_encoded = payment_encoder.transform(
    X_val[["Payment_of_Min_Amount"]]
)

# Noms des nouvelles colonnes
payment_columns = payment_encoder.get_feature_names_out(
    ["Payment_of_Min_Amount"]
)

print("Nombre de nouvelles colonnes :", len(payment_columns))
print("Noms des colonnes :", payment_columns)

Nombre de nouvelles colonnes : 3
Noms des colonnes : ['Payment_of_Min_Amount_NM' 'Payment_of_Min_Amount_No'
 'Payment_of_Min_Amount_Yes']



##### Intégration des variables encodées

Les trois variables binaires générées sont ajoutées aux ensembles d'entraînement et de validation.

La variable catégorielle originale `Payment_of_Min_Amount` est ensuite supprimée afin d'éviter de conserver simultanément sa représentation textuelle et sa représentation numérique.


In [28]:
# Création des DataFrames encodés
payment_train_df = pd.DataFrame(
    payment_train_encoded,
    columns=payment_columns,
    index=X_train.index
)

payment_val_df = pd.DataFrame(
    payment_val_encoded,
    columns=payment_columns,
    index=X_val.index
)

# Intégration dans les ensembles
X_train = pd.concat(
    [X_train.drop(columns=["Payment_of_Min_Amount"]), payment_train_df],
    axis=1
)

X_val = pd.concat(
    [X_val.drop(columns=["Payment_of_Min_Amount"]), payment_val_df],
    axis=1
)

# Mise à jour des variables catégorielles
categorical_features.remove("Payment_of_Min_Amount")

# Vérification
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)
print("Variables catégorielles restantes :", categorical_features)

Dimensions de X_train : (80000, 50)
Dimensions de X_val   : (20000, 50)
Variables catégorielles restantes : ['Credit_Mix', 'Payment_Behaviour']



##### Intégration des variables encodées

Les trois variables binaires générées sont ajoutées aux ensembles d'entraînement et de validation.

La variable catégorielle originale `Payment_of_Min_Amount` est ensuite supprimée afin d'éviter de conserver simultanément sa représentation textuelle et sa représentation numérique.


In [30]:
print("Payment_of_Min_Amount existe dans X_train :",
      "Payment_of_Min_Amount" in X_train.columns)

print("Colonnes encodées présentes :",
      [col for col in payment_columns if col in X_train.columns])

print("\nDimensions de X_train :", X_train.shape)
print("Dimensions de X_val :", X_val.shape)

print("\nVariables catégorielles :", categorical_features)

Payment_of_Min_Amount existe dans X_train : False
Colonnes encodées présentes : ['Payment_of_Min_Amount_NM', 'Payment_of_Min_Amount_No', 'Payment_of_Min_Amount_Yes']

Dimensions de X_train : (80000, 50)
Dimensions de X_val : (20000, 50)

Variables catégorielles : ['Credit_Mix', 'Payment_Behaviour']


In [31]:
print("Modalités dans l'entraînement :")
print(X_train["Payment_Behaviour"].value_counts())

print("\nModalités dans la validation :")
print(X_val["Payment_Behaviour"].value_counts())

Modalités dans l'entraînement :
Payment_Behaviour
Low_spent_Small_value_payments      20470
High_spent_Medium_value_payments    14057
Low_spent_Medium_value_payments     11101
High_spent_Large_value_payments     10931
High_spent_Small_value_payments      9034
Low_spent_Large_value_payments       8309
Inconnu                              6098
Name: count, dtype: int64

Modalités dans la validation :
Payment_Behaviour
Low_spent_Small_value_payments      5043
High_spent_Medium_value_payments    3483
High_spent_Large_value_payments     2790
Low_spent_Medium_value_payments     2760
High_spent_Small_value_payments     2306
Low_spent_Large_value_payments      2116
Inconnu                             1502
Name: count, dtype: int64



##### Transformation de Payment_Behaviour

Nous utilisons le One-Hot Encoding pour transformer les sept modalités de `Payment_Behaviour` en variables binaires indépendantes.

L'encodeur est ajusté sur l'ensemble d'entraînement, puis appliqué à l'ensemble de validation.


In [32]:
# Création de l'encodeur
behaviour_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apprentissage et transformation sur l'entraînement
behaviour_train_encoded = behaviour_encoder.fit_transform(
    X_train[["Payment_Behaviour"]]
)

# Transformation de la validation
behaviour_val_encoded = behaviour_encoder.transform(
    X_val[["Payment_Behaviour"]]
)

# Récupération des noms des nouvelles colonnes
behaviour_columns = behaviour_encoder.get_feature_names_out(
    ["Payment_Behaviour"]
)

print("Nombre de nouvelles colonnes :", len(behaviour_columns))
print("Noms des colonnes :")
print(behaviour_columns)

Nombre de nouvelles colonnes : 7
Noms des colonnes :
['Payment_Behaviour_High_spent_Large_value_payments'
 'Payment_Behaviour_High_spent_Medium_value_payments'
 'Payment_Behaviour_High_spent_Small_value_payments'
 'Payment_Behaviour_Inconnu'
 'Payment_Behaviour_Low_spent_Large_value_payments'
 'Payment_Behaviour_Low_spent_Medium_value_payments'
 'Payment_Behaviour_Low_spent_Small_value_payments']



##### Intégration des variables encodées

Les sept variables binaires générées sont ajoutées aux ensembles d'entraînement et de validation.

La variable catégorielle originale `Payment_Behaviour` est supprimée, car elle est désormais représentée par les nouvelles variables numériques.


In [33]:
# Création des DataFrames encodés
behaviour_train_df = pd.DataFrame(
    behaviour_train_encoded,
    columns=behaviour_columns,
    index=X_train.index
)

behaviour_val_df = pd.DataFrame(
    behaviour_val_encoded,
    columns=behaviour_columns,
    index=X_val.index
)

# Intégration dans les ensembles
X_train = pd.concat(
    [X_train.drop(columns=["Payment_Behaviour"]), behaviour_train_df],
    axis=1
)

X_val = pd.concat(
    [X_val.drop(columns=["Payment_Behaviour"]), behaviour_val_df],
    axis=1
)

# Mise à jour des variables catégorielles
categorical_features.remove("Payment_Behaviour")

# Vérification
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)
print("Variables catégorielles restantes :", categorical_features)

Dimensions de X_train : (80000, 56)
Dimensions de X_val   : (20000, 56)
Variables catégorielles restantes : ['Credit_Mix']


In [34]:
# Credit_Mix a déjà été encodée en valeurs numériques
categorical_features.remove("Credit_Mix")

print("Variables catégorielles restantes :", categorical_features)
print("Nombre de variables numériques :",
      len(X_train.select_dtypes(include=["int64", "float64"]).columns))


Variables catégorielles restantes : []
Nombre de variables numériques : 56



#### 2.6. Normalisation des variables numériques

Les variables numériques présentent des échelles et des amplitudes différentes. Par exemple, l'âge et le revenu annuel ne possèdent pas les mêmes ordres de grandeur.

Nous appliquons une normalisation afin de rendre les variables quantitatives comparables, notamment pour les algorithmes sensibles aux distances comme KNN et SVM.

Les variables binaires et cycliques ne sont pas concernées par cette opération.


In [35]:
# Les variables numériques originales à normaliser
numerical_features_to_scale = numerical_features.copy()

print("Nombre de variables à normaliser :",
      len(numerical_features_to_scale))

print("\nVariables concernées :")
print(numerical_features_to_scale)

Nombre de variables à normaliser : 17

Variables concernées :
['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']


#### 2.6.1. Application de StandardScaler

La méthode StandardScaler transforme chaque variable selon la formule :

\[
Z = \frac{X - \mu}{\sigma}
\]

où :
- \(X\) : valeur initiale ;
- \(\mu\) : moyenne calculée sur les données d'entraînement ;
- \(\sigma\) : écart-type calculé sur les données d'entraînement.

Le scaler est ajusté uniquement sur l'ensemble d'entraînement, puis appliqué à l'ensemble de validation afin d'éviter toute fuite de données (*data leakage*).

In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Ajustement et transformation sur les données d'entraînement
X_train[numerical_features_to_scale] = scaler.fit_transform(
    X_train[numerical_features_to_scale]
)

# Transformation des données de validation avec le même scaler
X_val[numerical_features_to_scale] = scaler.transform(
    X_val[numerical_features_to_scale]
)

In [37]:
# Vérifier les statistiques des variables après normalisation
X_train[numerical_features_to_scale].describe().loc[
    ["mean", "std", "min", "max"]
].T

,mean,std,min,max
Age,-9.379164e-17,1.000006,-1.816368,6.181979
Annual_Income,1.083578e-17,1.000006,-0.129521,56.054175
Monthly_Inhand_Salary,-1.126210e-16,1.000006,-1.257209,3.779064
Num_Bank_Accounts,2.806644e-17,1.000006,-2.084610,2.187779
Num_Credit_Card,-1.110223e-16,1.000006,-2.695897,2.677384
Interest_Rate,8.668621e-17,1.000006,-1.561592,2.253111
Num_of_Loan,3.304024e-17,1.000006,-1.463111,2.291357
Delay_from_due_date,-1.136868e-17,1.000006,-1.755736,3.091079
Num_of_Delayed_Payment,-9.769963e-17,1.000006,-2.264548,2.441949
Changed_Credit_Limit,6.181722e-17,1.000006,-2.511776,3.962616


In [38]:
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_val   :", X_val.shape)

print("\nValeurs manquantes dans X_train :", X_train.isnull().sum().sum())
print("Valeurs manquantes dans X_val   :", X_val.isnull().sum().sum())

print("\nTypes de données :")
print(X_train.dtypes.value_counts())

Dimensions de X_train : (80000, 56)
Dimensions de X_val   : (20000, 56)

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_val   : 0

Types de données :
float64    45
int64      11
Name: count, dtype: int64


### 6. Exportation des données prétraitées

Les ensembles d'entraînement et de validation, après nettoyage, imputation,
encodage et normalisation, sont exportés au format CSV afin d'être réutilisés
dans la phase de modélisation.

In [39]:
# Exportation des données prétraitées

X_train.to_csv("X_train_preprocessed.csv", index=False)
X_val.to_csv("X_val_preprocessed.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)

print("Exportation terminée avec succès.")

Exportation terminée avec succès.
